# 11 Enterprise Research Report Generator

In [19]:
# =====================================================
# Notebook 11
# Enterprise Research Report Generator
# =====================================================

from typing import TypedDict
from typing import List
from typing import Dict

from transformers import pipeline

In [20]:
llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_new_tokens=1024,
    temperature=0.3,
)

Device set to use cpu


In [21]:
class EnterpriseResearchState(TypedDict):

    research_goal: str

    findings: List[str]

    conflicts: List[Dict]

    trust_score: float

    compressed_memory: str

    final_report: str

    status: str

    metadata: Dict

In [22]:
state = {
    "research_goal": "Compare cloud infrastructure costs against competitors",
    "findings": [
        "AWS pricing increased 8%",
        "Azure pricing remained stable",
        "Google Cloud storage costs reduced 4%",
        "Oracle Cloud expanded GPU offerings",
        "Vendor lock-in risk identified",
        "GDPR review required",
    ],
    "conflicts": [],
    "trust_score": 0.92,
    "compressed_memory": "",
    "final_report": "",
    "status": "",
    "metadata": {},
}

In [23]:
def build_report_prompt(state):

    findings = "\n".join(state["findings"])

    prompt = f"""
You are a Senior Enterprise Strategy Consultant.

Research Goal:
{state['research_goal']}

Findings:
{findings}

Trust Score:
{state['trust_score']}

Create a professional report with:

1. Executive Summary

2. Key Findings

3. Risk Assessment

4. Strategic Recommendations

5. Confidence Assessment

Use clear business language.
"""

    return prompt

In [24]:
def generate_report(state):

    prompt = build_report_prompt(state)

    result = llm(prompt)

    return result[0]["generated_text"]

In [25]:
report = generate_report(state)

print(report)


You are a Senior Enterprise Strategy Consultant.

Research Goal:
Compare cloud infrastructure costs against competitors

Findings:
AWS pricing increased 8%
Azure pricing remained stable
Google Cloud storage costs reduced 4%
Oracle Cloud expanded GPU offerings
Vendor lock-in risk identified
GDPR review required

Trust Score:
0.92

Create a professional report with:

1. Executive Summary

2. Key Findings

3. Risk Assessment

4. Strategic Recommendations

5. Confidence Assessment

Use clear business language.
**Executive Summary**

This report provides an in-depth analysis of the current and projected cost landscape for various cloud providers, including Amazon Web Services (AWS), Microsoft Azure, Google Cloud Platform (GCP), and Oracle Cloud Infrastructure (OCI). The findings reveal significant variations in pricing across these platforms, with AWS experiencing the highest increase in costs due to market competition pressures. Conversely, GCP has seen its storage costs decrease by 4%, w

In [26]:
def writer_agent(state):

    print("\nWriter Agent Running...")

    report = generate_report(state)

    state["final_report"] = report

    state["status"] = "report_generated"

    return state

In [27]:
updated_state = writer_agent(state)


Writer Agent Running...


In [28]:
print(updated_state["final_report"])


You are a Senior Enterprise Strategy Consultant.

Research Goal:
Compare cloud infrastructure costs against competitors

Findings:
AWS pricing increased 8%
Azure pricing remained stable
Google Cloud storage costs reduced 4%
Oracle Cloud expanded GPU offerings
Vendor lock-in risk identified
GDPR review required

Trust Score:
0.92

Create a professional report with:

1. Executive Summary

2. Key Findings

3. Risk Assessment

4. Strategic Recommendations

5. Confidence Assessment

Use clear business language.
Sure, I can help you create a professional report based on the provided information and research goals. Here's an outline of what we'll cover in each section:

### Executive Summary
- **Objective**: To compare cloud infrastructure costs among AWS, Azure, Google Cloud, and Oracle Cloud to identify potential cost savings or risks associated with vendor lock-in.
- **Key Points**:
  - AWS prices have increased by 8%.
  - Azure prices remain stable.
  - Google Cloud has seen a reduction 

In [29]:
def build_enterprise_prompt(state):

    findings = "\n".join(state["findings"])

    return f"""
ROLE:
Senior Enterprise Research Consultant

GOAL:
{state['research_goal']}

EVIDENCE:
{findings}

TRUST SCORE:
{state['trust_score']}

OUTPUT FORMAT:

# Executive Summary

# Key Findings

# Risks

# Recommendations

# Confidence Assessment

Rules:

- Use only supplied evidence.
- Do not invent facts.
- Mention uncertainties.
- Be concise.
"""

In [30]:
def extract_risks(findings):

    keywords = ["risk", "issue", "concern", "compliance", "vendor lock-in", "gdpr"]

    risks = []

    for finding in findings:

        for keyword in keywords:

            if keyword.lower() in finding.lower():

                risks.append(finding)

                break

    return risks

In [31]:
extract_risks(state["findings"])

['Vendor lock-in risk identified', 'GDPR review required']

In [32]:
state["metadata"]["citations"] = [
    {
        "finding": "AWS pricing increased 8%",
        "source": "Cloud_Report_2026.pdf",
        "page": 12,
    },
    {
        "finding": "Azure pricing remained stable",
        "source": "Azure_Analysis.pdf",
        "page": 5,
    },
]

In [33]:
def format_citations(citations):

    lines = []

    for item in citations:

        lines.append(
            f"{item['finding']} " f"(Source: {item['source']}, " f"Page {item['page']})"
        )

    return "\n".join(lines)

In [34]:
print(format_citations(state["metadata"]["citations"]))

AWS pricing increased 8% (Source: Cloud_Report_2026.pdf, Page 12)
Azure pricing remained stable (Source: Azure_Analysis.pdf, Page 5)


In [35]:
def executive_summary_agent(state):

    prompt = f"""
Create a concise executive summary.

Findings:

{chr(10).join(state['findings'])}
"""

    result = llm(prompt)

    return result[0]["generated_text"]

In [36]:
summary = executive_summary_agent(state)

print(summary)


Create a concise executive summary.

Findings:

AWS pricing increased 8%
Azure pricing remained stable
Google Cloud storage costs reduced 4%
Oracle Cloud expanded GPU offerings
Vendor lock-in risk identified
GDPR review required
Data privacy concerns raised

Recommendations:
1. Monitor AWS and Azure prices closely.
2. Evaluate Google Cloud storage cost reduction.
3. Investigate Oracle Cloud GPU expansion.
4. Review GDPR compliance status.
5. Address data privacy issues with customers.

Next steps: 
- Conduct quarterly price comparisons for AWS and Azure.
- Analyze Google Cloud storage cost savings.
- Assess Oracle Cloud GPU capabilities.
- Update GDPR policies as needed.
- Communicate with affected customers about data privacy measures.

This executive summary provides a clear overview of the key findings, recommendations, and next steps for managing cloud computing vendor risks and ensuring customer trust in our technology solutions. It highlights areas where we can improve to mitiga

In [37]:
from langgraph.graph import StateGraph, END

In [38]:
graph = StateGraph(EnterpriseResearchState)

In [39]:
graph.add_node("writer", writer_agent)

In [40]:
graph.set_entry_point("writer")

graph.add_edge("writer", END)

In [41]:
app = graph.compile()

In [42]:
result = app.invoke(state)


Writer Agent Running...
